# Lab 02.3 — Ottimizzazione delle query

Come Spark esegue le query in modo efficiente:

1. shuffle e numero di partizioni, **Adaptive Query Execution (AQE)**
2. strategie di **join** (broadcast vs sort-merge)
3. `repartition` vs `coalesce`
4. **caching** dei DataFrame

In [ ]:
import os, time
from pyspark.sql import SparkSession, functions as F
import dati_shop

spark = (SparkSession.builder
         .appName("Lab02-Ottimizzazione")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .getOrCreate())
t = dati_shop.carica(spark)
ordini, prodotti, clienti = t["ordini"], t["prodotti"], t["clienti"]

def cronometra(etichetta, azione):
    t0 = time.time(); r = azione(); print(f"{etichetta:45s} {time.time() - t0:6.2f}s"); return r

## 1. Shuffle e numero di partizioni
Dopo uno shuffle (`groupBy`, `join`, `orderBy`...) il numero di partizioni è `spark.sql.shuffle.partitions` (default **200**).
Con pochi dati, 200 task minuscoli costano più del calcolo stesso.
**AQE** misura i dati reali dopo lo shuffle e **unisce** le partizioni troppo piccole.

In [ ]:
query = lambda: ordini.groupBy("cliente_id").agg(F.sum("quantita").alias("q"))

spark.conf.set("spark.sql.adaptive.enabled", "false")
df = query(); cronometra("AQE OFF", df.collect)
print("  partizioni risultato:", df.rdd.getNumPartitions())

spark.conf.set("spark.sql.adaptive.enabled", "true")
df = query(); cronometra("AQE ON", df.collect)
print("  partizioni risultato:", df.rdd.getNumPartitions(), "(unite da AQE: 'AQEShuffleRead coalesced' nel piano finale)")

## 2. Strategie di join
* **Broadcast Hash Join**: la tabella piccola viene copiata su ogni executor → nessuno shuffle della tabella grande.
* **Sort-Merge Join**: entrambe le tabelle vengono ripartizionate (shuffle) e ordinate per chiave → adatto a due tabelle grandi.

Spark sceglie in base a `spark.sql.autoBroadcastJoinThreshold` (10 MB); si può forzare con gli *hint*.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "false")   # per confrontare le strategie "pure"

smj = ordini.join(prodotti.hint("merge"), "prodotto_id").groupBy("categoria").count()
bhj = ordini.join(F.broadcast(prodotti), "prodotto_id").groupBy("categoria").count()

cronometra("Sort-Merge Join", smj.collect)
cronometra("Broadcast Hash Join", bhj.collect)
smj.explain()
bhj.explain()
spark.conf.set("spark.sql.adaptive.enabled", "true")

## 3. `repartition` vs `coalesce`
* `repartition(n)` → shuffle completo, partizioni bilanciate (può aumentare o diminuire n).
* `coalesce(n)` → unisce partizioni esistenti **senza shuffle** (solo per diminuire n), utile prima di scrivere pochi file.

In [ ]:
molte = ordini.repartition(16)
print("iniziali              :", ordini.rdd.getNumPartitions())
print("repartition(16)       :", molte.rdd.getNumPartitions())
print("repartition(16) → coalesce(4):", molte.coalesce(4).rdd.getNumPartitions())
molte.coalesce(4).explain()      # un solo 'Exchange' (quello del repartition): coalesce non fa shuffle

## 4. Caching di DataFrame
Un DataFrame riutilizzato più volte (es. in analisi interattive) può essere tenuto in memoria in formato colonnare compresso.

In [ ]:
arricchiti = ordini.join(prodotti, "prodotto_id").withColumn("importo", F.col("quantita") * F.col("prezzo"))

cronometra("senza cache - query 1", lambda: arricchiti.groupBy("categoria").sum("importo").collect())
cronometra("senza cache - query 2", lambda: arricchiti.groupBy("canale").sum("importo").collect())
arricchiti.cache().count()
cronometra("con cache   - query 1", lambda: arricchiti.groupBy("categoria").sum("importo").collect())
cronometra("con cache   - query 2", lambda: arricchiti.groupBy("canale").sum("importo").collect())
arricchiti.unpersist()

In [ ]:
spark.stop()